## Tutorial: Concurrent OpenAI API Calls with `asyncio`

This notebook demonstrates how to use Python's `asyncio` library and the `AsyncOpenAI` client to execute multiple non-blocking LLM requests concurrently.

### Key Concepts Covered:
* **Asynchronous Execution:** Instead of waiting for API responses sequentially (which multiplies your total wait time), `asyncio` handles multiple network requests in parallel.
* **Performance Boost:** Total execution time drops from the sum of all requests down to the duration of the *slowest* individual request.
* **Jupyter Note:** Because Jupyter runs its own background event loop, avoid using `asyncio.run()` here. Instead, execute your async functions directly using `await main()`.

In [6]:
# Import asyncio to handle asynchronous programming and event loops
import asyncio

# Import time to measure execution duration and track timestamps
import time

# Import AsyncOpenAI to make non-blocking concurrent requests to OpenAI APIs
from openai import AsyncOpenAI, OpenAI

In [7]:
client = AsyncOpenAI()

In [8]:
async def ask_llm(prompt_id: int, topic: str):
    """
    Asynchronously sends a single prompt to OpenAI's Chat Completions API.
    
    Parameters:
    - prompt_id (int): Identifier to track request execution order.
    - topic (str): The subject matter for the LLM prompt.
    
    Returns:
    - dict: Containing prompt metadata, duration, and the generated LLM response.
    """
    # Log start timestamp and request details
    print(f"[{time.strftime('%X')}] Request {prompt_id}: Sending prompt about '{topic}'...")
    
    start_time = time.time()
    
    # Non-blocking API call using the AsyncOpenAI client.
    # The 'await' keyword pauses execution of this specific coroutine, yielding control 
    # back to the event loop so other pending tasks/requests can proceed concurrently.
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Keep your responses under 20 words."},
            {"role": "user", "content": f"Give me a fun fact about {topic}."}
        ],
        temperature=0.7,
    )
    
    # Calculate response round-trip time
    duration = time.time() - start_time
    content = response.choices[0].message.content.strip()
    
    print(f"[{time.strftime('%X')}] Request {prompt_id}: Received response in {duration:.2f}s")
    
    # Return structured result dictionary
    return {"id": prompt_id, "topic": topic, "response": content}


In [9]:
async def main():
    """
    Main orchestration coroutine that schedules and concurrently executes
    multiple LLM API requests using asyncio.
    """
    print(f"[{time.strftime('%X')}] Starting concurrent OpenAI API calls...\n")
    
    total_start = time.time()

    # List of topics to query in parallel
    topics = ["quantum computing", "ocean exploration", "space travel", "coffee history"]

    # Wrap each coroutine call in an asyncio Task.
    # asyncio.create_task() schedules the coroutine to run on the active event loop immediately.
    tasks = [
        asyncio.create_task(ask_llm(prompt_id=i + 1, topic=topic))
        for i, topic in enumerate(topics)
    ]

    # asyncio.gather() unpacks task objects, executes them concurrently, 
    # and waits until all tasks complete before returning their aggregated results.
    # When you place an asterisk in front of it (*tasks), Python unpacks the items 
    # out of the list container and passes them as separate, individual positional arguments.
    results = await asyncio.gather(*tasks)

    # Calculate total duration for all concurrent requests
    total_duration = time.time() - total_start
    
    # Print runtime summary and retrieved results
    print("\n" + "=" * 50)
    print(f"[{time.strftime('%X')}] All requests finished in {total_duration:.2f} seconds!")
    print("=" * 50 + "\n")

    for item in results:
        print(f"🔹 Prompt {item['id']} ({item['topic']}):\n   {item['response']}\n")

In [ ]:
# Await the main coroutine directly (used in Jupyter Notebooks or Colab 
# where an event loop is already running in the background)
# Note: In a standalone .py script file, you would run this instead using await: 
# asyncio.run(main())
await main()

[05:21:37] Starting concurrent OpenAI API calls...

[05:21:37] Request 1: Sending prompt about 'quantum computing'...
[05:21:37] Request 2: Sending prompt about 'ocean exploration'...
[05:21:37] Request 3: Sending prompt about 'space travel'...
[05:21:37] Request 4: Sending prompt about 'coffee history'...


[05:21:38] Request 1: Received response in 0.77s
[05:21:38] Request 2: Received response in 0.78s
[05:21:38] Request 4: Received response in 0.88s
[05:21:38] Request 3: Received response in 1.04s

[05:21:38] All requests finished in 1.05 seconds!

🔹 Prompt 1 (quantum computing):
   Quantum computers can solve certain problems much faster than classical computers, like factoring large numbers.

🔹 Prompt 2 (ocean exploration):
   More than 80% of the ocean is unexplored and unmapped!

🔹 Prompt 3 (space travel):
   Astronauts can grow up to 2 inches taller in space due to spinal expansion!

🔹 Prompt 4 (coffee history):
   Coffee was discovered in Ethiopia in the 9th century by a goat herder named Kaldi.



### **Key Takeaway: The Power of `asyncio`**

In a traditional synchronous program, sending multiple AI requests one after another forces your code to block and wait sequentially, stacking up delays. By leveraging `asyncio` and `AsyncOpenAI`, multiple API calls are dispatched simultaneously through cooperative multitasking without blocking the main execution thread. As a result, the total execution time drops dramatically from the sum of all individual delays down to just the duration of the single slowest request. This example highlights how asynchronous programming transforms sluggish network tasks into high-throughput, efficient operations ideal for modern AI applications.